In [1]:
import argparse
import numpy as np
import pandas as pd
import pvlib
import pytz
import laspy
import plotly.graph_objects as go

# ============================================================================
# CONFIG
# ============================================================================

DEFAULT_LIDAR      = "output/recovered.laz"
DEFAULT_SHADOW_CSV = "results/shadow_matrix_results_SE_pro/shadow_attenuation_matrix_conecasting_SE_v10.csv"

LAT, LON       = 62.979849, 27.648656
LOCAL_TZ       = "Europe/Helsinki"

ARRAY_CORNER_XY    = np.array([532882.00, 6983507.00])
ROOF_SEARCH_RADIUS = 3.0
OFFSET_FROM_ROOF   = -0.5
TILT_DEG, AZ_DEG   = 12, 170
PANEL_W, PANEL_H   = 1.0, 1.6
ROW_CONFIG         = (5, 4, 3)

DEFAULT_RADIUS  = 50
RAY_LENGTH      = 80
POINT_SUBSAMPLE = 3

DEFAULT_VOXEL_SIZE    = 2.0
DEFAULT_VOXEL_OPACITY = 0.0
DEFAULT_RAY_CONE_ANGLE_DEG = 5.0
DEFAULT_RAY_CONE_SEGMENTS = 8
# Edge length of the cube voxels representing occupied space. Adjust based on LiDAR density and desired visual clarity.


CLASS_COLORS = {
    2: ("Ground",   "rgb(180,150,120)"),
    3: ("Low Veg",  "rgb(144,210,144)"),
    4: ("Med Veg",  "rgb(60,160,60)"),
    5: ("High Veg", "rgb(20,100,20)"),
    6: ("Building", "rgb(200,60,60)"),
}

_CLASS_PRIORITY = {6: 5, 5: 4, 4: 3, 3: 2, 2: 1}

# ============================================================================
# HELPERS
# ============================================================================

def generate_pv_array_points(corner_coords):
    tilt_rad   = np.radians(TILT_DEG)
    rot_z_rad  = np.radians(180 - AZ_DEG)
    total_h    = (len(ROW_CONFIG) - 1) * PANEL_H
    y_steps    = np.linspace(total_h, 0.0, len(ROW_CONFIG))
    pts = []
    for i, n in enumerate(ROW_CONFIG):
        for p in range(n):
            pts.append([p * PANEL_W + PANEL_W / 2, y_steps[i], 0.0])
    pts = np.array(pts)
    Rt = np.array([[1,0,0],[0,np.cos(tilt_rad),-np.sin(tilt_rad)],[0,np.sin(tilt_rad),np.cos(tilt_rad)]])
    Rz = np.array([[np.cos(rot_z_rad),-np.sin(rot_z_rad),0],[np.sin(rot_z_rad),np.cos(rot_z_rad),0],[0,0,1]])
    return (Rz @ Rt @ pts.T).T + corner_coords


from pv_analysis_re import load_and_smooth_shadow_matrix

def lookup_shadow(shadow_csv, altitude_deg, azimuth_deg):
    sm = load_and_smooth_shadow_matrix(shadow_csv, window_size=(2, 2))
    if altitude_deg < 0.5:
        return 0.0, 1.0
    ai = int(np.clip(np.round(altitude_deg), 0, sm.shape[0] - 1))
    zi = int(np.clip(np.round(azimuth_deg),  0, sm.shape[1] - 1))
    sf = float(np.clip(sm[ai, zi], 0.0, 1.0))
    return sf, 1.0 - sf


# ============================================================================
# VOXEL REPRESENTATION
# ============================================================================

def voxelize_for_viz(points, classifications, voxel_size,
                     scene_min=None, scene_max=None):
    """Sparse voxelisation: occupied voxel min-corners, dominant class, point count."""
    if scene_min is None:
        scene_min = np.min(points, axis=0)
    if scene_max is None:
        scene_max = np.max(points, axis=0)
    grid_dims = np.maximum(
        np.ceil((scene_max - scene_min) / voxel_size).astype(np.int64), 1
    )
    nx, ny, nz = grid_dims

    vi = np.clip(
        np.floor((points - scene_min) / voxel_size).astype(np.int64),
        0, grid_dims - 1,
    )
    keys = vi[:, 0] * (ny * nz) + vi[:, 1] * nz + vi[:, 2]

    pri_lut = np.zeros(int(max(_CLASS_PRIORITY) + 1) + 1, dtype=np.int32)
    for c, p in _CLASS_PRIORITY.items():
        pri_lut[c] = p
    cls_clipped = np.clip(classifications, 0, len(pri_lut) - 1).astype(np.int64)
    pri = pri_lut[cls_clipped]

    order = np.lexsort((-pri, keys))
    keys_sorted = keys[order]
    cls_sorted  = classifications[order]

    uniq_keys, first_idx = np.unique(keys_sorted, return_index=True)
    dominant_cls = cls_sorted[first_idx]

    counts = np.bincount(np.searchsorted(uniq_keys, keys),
                         minlength=len(uniq_keys)).astype(np.int32)

    ix = uniq_keys // (ny * nz)
    iy = (uniq_keys // nz) % ny
    iz = uniq_keys % nz
    voxel_min = (np.stack([ix, iy, iz], axis=1).astype(np.float64)
                 * voxel_size + scene_min)
    return voxel_min, dominant_cls, counts


_CUBE_VERTS = np.array([
    [0, 0, 0], [1, 0, 0], [1, 1, 0], [0, 1, 0],
    [0, 0, 1], [1, 0, 1], [1, 1, 1], [0, 1, 1],
], dtype=np.float64)
_CUBE_I = np.array([0, 0, 4, 4, 0, 0, 3, 3, 0, 0, 1, 1])
_CUBE_J = np.array([1, 2, 5, 6, 1, 5, 2, 6, 3, 7, 2, 6])
_CUBE_K = np.array([2, 3, 6, 7, 5, 4, 6, 7, 7, 4, 6, 5])


def make_voxel_mesh_traces(voxel_min, voxel_classes, voxel_size,
                            class_colors=CLASS_COLORS,
                            opacity=DEFAULT_VOXEL_OPACITY,
                            visible_classes=None):
    """One go.Mesh3d per class: every occupied voxel becomes a cube."""
    traces = []
    cube_v = _CUBE_VERTS * voxel_size
    visible_classes = set(class_colors) if visible_classes is None else set(visible_classes)

    for cid, (label, color) in class_colors.items():
        if cid not in visible_classes:
            continue
        mask = voxel_classes == cid
        n = int(mask.sum())
        if n == 0:
            continue

        mins  = voxel_min[mask]
        verts = (mins[:, None, :] + cube_v[None, :, :]).reshape(-1, 3)

        offsets = (np.arange(n) * 8).reshape(-1, 1)
        i_idx = (_CUBE_I[None, :] + offsets).ravel()
        j_idx = (_CUBE_J[None, :] + offsets).ravel()
        k_idx = (_CUBE_K[None, :] + offsets).ravel()

        traces.append(go.Mesh3d(
            x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
            i=i_idx, j=j_idx, k=k_idx,
            color=color, opacity=opacity, flatshading=True,
            name=f"{label} voxels ({n:,})",
            showlegend=True, hoverinfo="skip",
        ))
    return traces


# ============================================================================
# MAIN
# ============================================================================

def visualize_scene(local_time_str, lidar_path=DEFAULT_LIDAR,
                    shadow_csv=DEFAULT_SHADOW_CSV, radius=DEFAULT_RADIUS,
                    ray_length=RAY_LENGTH, subsample=POINT_SUBSAMPLE,
                    show_points=True, show_voxels=False,
                    voxel_size=DEFAULT_VOXEL_SIZE,
                    voxel_opacity=DEFAULT_VOXEL_OPACITY,
                    voxel_classes=None):

    local_time  = pd.Timestamp(local_time_str)
    tz          = pytz.timezone(LOCAL_TZ)
    utc_time    = tz.localize(local_time.to_pydatetime()).astimezone(pytz.UTC)

    solpos      = pvlib.solarposition.get_solarposition(pd.DatetimeIndex([utc_time]), LAT, LON)
    alt_deg     = 90.0 - float(solpos["apparent_zenith"].iloc[0])
    azi_deg     = float(solpos["azimuth"].iloc[0])

    if alt_deg < 0.5:
        print("Sun below horizon.")
        return

    el, az  = np.radians(alt_deg), np.radians(azi_deg)
    sun_dir = np.array([np.cos(el)*np.sin(az), np.cos(el)*np.cos(az), np.sin(el)])

    las = laspy.read(lidar_path)
    pts = np.vstack((las.x, las.y, las.z)).T
    cls = np.array(las.classification)
    mask = np.isin(cls, [2, 3, 4, 5, 6])
    pts, cls = pts[mask], cls[mask]

    dists     = np.linalg.norm(pts[:, :2] - ARRAY_CORNER_XY, axis=1)
    roof_mask = (dists < ROOF_SEARCH_RADIUS) & (cls == 6)
    target_z  = (np.max(pts[roof_mask, 2]) + OFFSET_FROM_ROOF) if roof_mask.any() \
                else np.median(pts[cls == 2, 2]) + 5.0

    panel_pts    = generate_pv_array_points(np.array([*ARRAY_CORNER_XY, target_z]))
    array_center = panel_pts.mean(axis=0)

    crop  = np.linalg.norm(pts[:, :2] - array_center[:2], axis=1) < radius
    pts_crop, cls_crop = pts[crop], cls[crop]
    pts_s = pts_crop[::subsample]
    cls_s = cls_crop[::subsample]

    shadow_factor, transmittance = lookup_shadow(shadow_csv, alt_deg, azi_deg)

    # ---- Traces ----
    traces = []

    if show_points:
        for cid, (label, color) in CLASS_COLORS.items():
            m = cls_s == cid
            if not m.any():
                continue
            p = pts_s[m]
            traces.append(go.Scatter3d(
                x=p[:,0], y=p[:,1], z=p[:,2],
                mode="markers",
                marker=dict(size=1.2, color=color, opacity=0.55),
                name=label,
                hoverinfo="skip",
            ))

    if show_voxels:
        v_min, v_cls, _ = voxelize_for_viz(pts_crop, cls_crop, voxel_size)
        n_total = len(v_min)
        if voxel_classes is not None:
            keep = np.isin(v_cls, list(voxel_classes))
            v_min, v_cls = v_min[keep], v_cls[keep]
        print(f"  Voxelisation: {n_total:,} occupied voxels @ {voxel_size}m "
              f"(showing {len(v_min):,})")
        traces.extend(make_voxel_mesh_traces(
            v_min, v_cls, voxel_size,
            class_colors=CLASS_COLORS, opacity=voxel_opacity,
            visible_classes=voxel_classes,
        ))

    traces.append(go.Scatter3d(
        x=panel_pts[:,0], y=panel_pts[:,1], z=panel_pts[:,2],
        mode="markers",
        marker=dict(size=7, color="rgb(255,210,0)", symbol="diamond",
                    line=dict(width=1, color="rgb(180,130,0)")),
        name="PV panels",
        hoverinfo="skip",
    ))

    for i, pt in enumerate(panel_pts):
        end = pt + sun_dir * ray_length
        traces.append(go.Scatter3d(
            x=[pt[0], end[0]], y=[pt[1], end[1]], z=[pt[2], end[2]],
            mode="lines",
            line=dict(width=3, color="rgba(255,160,0,0.85)"),
            name="Solar rays" if i == 0 else None,
            showlegend=(i == 0),
            hoverinfo="skip",
        ))

    sun_pos = array_center + sun_dir * ray_length * 1.15
    traces.append(go.Scatter3d(
        x=[sun_pos[0]], y=[sun_pos[1]], z=[sun_pos[2]],
        mode="markers",
        marker=dict(size=14, color="rgb(255,220,0)",
                    line=dict(width=2, color="rgb(220,140,0)")),
        name="Sun",
        hoverinfo="skip",
    ))

    # ---- Layout ----
    FF = "Arial, sans-serif"

    def axis(label):
        return dict(
            title=dict(text=label, font=dict(size=13, family=FF)),
            tickfont=dict(size=11, family=FF),
            showgrid=True, gridcolor="rgba(200,200,200,0.4)", gridwidth=1,
            showbackground=True, backgroundcolor="rgb(250,250,252)",
            zeroline=False,
        )

    ann = (f"Solar altitude: {alt_deg:.1f}° | "
           f"Azimuth: {azi_deg:.1f}° | "
           f"Transmittance: {transmittance:.3f} | "
           f"Shadow factor: {shadow_factor:.3f}")

    fig = go.Figure(data=traces)
    fig.update_layout(
        title=dict(
            text="3D LiDAR Scene with Cone-Cast Solar Ray Paths — Bank 1",
            font=dict(size=17, family=FF, color="rgb(40,40,40)"),
            x=0.5, xanchor="center",
        ),
        paper_bgcolor="white",
        scene=dict(
            xaxis=axis("Easting (m)"),
            yaxis=axis("Northing (m)"),
            zaxis=axis("Elevation (m)"),
            aspectmode="data",
            bgcolor="rgb(250,250,252)",
            camera=dict(eye=dict(x=1.5, y=-1.5, z=1.0), up=dict(x=0, y=0, z=1)),
        ),
        legend=dict(
            font=dict(size=12, family=FF),
            bgcolor="rgba(255,255,255,0.92)",
            bordercolor="rgba(150,150,150,0.5)",
            borderwidth=1,
            yanchor="top", y=0.97,
            xanchor="left", x=0.01,
            itemsizing="constant",
        ),
        annotations=[dict(
            text=ann, x=0.5, y=0.0, xref="paper", yref="paper",
            xanchor="center", yanchor="top", showarrow=False,
            font=dict(size=12, family=FF, color="rgb(80,80,80)"),
        )],
        margin=dict(l=0, r=0, t=60, b=50),
        width=1400, height=900,
    )

    html_path = f"scene_3d_{local_time.strftime('%Y%m%d_%H%M')}.html"
    fig.write_html(html_path)
    print(f"Saved to {html_path}")
    fig.show()
    return fig


In [2]:
args = argparse.Namespace(
    time="2021-07-26 15:00",
    lidar=DEFAULT_LIDAR,
    shadow=DEFAULT_SHADOW_CSV,
    radius=DEFAULT_RADIUS,
    ray_length=RAY_LENGTH,
    subsample=POINT_SUBSAMPLE,
)

fig = visualize_scene(
    args.time,
    lidar_path=args.lidar,
    shadow_csv=args.shadow,
    radius=args.radius,
    ray_length=args.ray_length,
    subsample=args.subsample,
    # --- voxel display ---
    show_points=False,        # set False to hide the raw LiDAR cloud
    show_voxels=True,        # toggle voxel cube rendering
    voxel_size=DEFAULT_VOXEL_SIZE,
    voxel_opacity=DEFAULT_VOXEL_OPACITY,
    voxel_classes=None,      # e.g. [5, 6] for high-veg + buildings only
    show_sun=True,           # toggle sun marker display
    show_rays=True,          # toggle solar ray line display
    ray_cone_angle_deg=DEFAULT_RAY_CONE_ANGLE_DEG,
    ray_cone_segments=DEFAULT_RAY_CONE_SEGMENTS,
)


TypeError: visualize_scene() got an unexpected keyword argument 'show_sun'